In [1]:
%reset -f

In [2]:
debug = False

In [3]:
 %reload_ext autoreload
%autoreload 2

import sys
sys.path.append("/home/zhuchen/poc/scorecard") 
FILE_PATH = '/home/zhuchen/poc/01-mashang/DeltaV1/file/'
DATA_PATH = '/home/zhuchen/poc/01-mashang/DeltaV1/data/'
TMP_PATH  = '/home/zhuchen/poc/01-mashang/DeltaV1/tmp/'
DATA_OR_PATH = '/home/zhuchen/poc/01-mashang/data_or/'

import os
import gc
import pandas as pd 
import numpy as np  
import math  
import zipfile
import matplotlib.pyplot as plt
import copy
import seaborn as sns
from pylab import mpl
from ScoreCard.creat_report import Report
mpl.rcParams['font.sans-serif'] = ['SimHei']
mpl.rcParams['axes.unicode_minus'] = False  

isExists=os.path.exists(FILE_PATH)
if not isExists:
    os.makedirs(FILE_PATH) 
    
isExists=os.path.exists(DATA_PATH)
if not isExists:
    os.makedirs(DATA_PATH)

isExists=os.path.exists(TMP_PATH)
if not isExists:
    os.makedirs(TMP_PATH)  

import warnings
warnings.filterwarnings("ignore")

## 1. 数据预处理

In [4]:
with zipfile.ZipFile(DATA_OR_PATH + '驭鉴20250707_LH_deltaV1_result.txt.zip', 'r') as zip_file:
    file_list = zip_file.namelist()
    print("压缩包内文件:", file_list)
    csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]
    with zip_file.open(csv_filename) as csv_file:
        data_tmp = pd.read_csv(csv_file, nrows=100)

压缩包内文件: ['Θ⌐¡Θë┤20250707_LH_deltaV1_result.txt']


In [5]:
data_tmp

,name,mobile,idCard,reqToken,backPointTime,TZ_0000_m1,TZ_0000_m12,TZ_0000_m15,TZ_0000_m18,TZ_0000_m2,...,TZ_2024_T4_m4,TZ_2024_T4_m5,TZ_2024_T4_m6,TZ_2024_T4_m7,TZ_2024_T4_m8,TZ_2024_T4_m9,TZ_2025_T1_m24,TZ_2025_T2_m24,TZ_2025_T3_m24,TZ_2025_T4_m24
0,test,d45735e2015b20c4e8f95645096ae0dac3c9924b9e6b35...,test,2618468494244577696,2024/11/21,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,test,dff94df987f14dd7304d25f291729f0b4926adad0df39b...,test,2626479791808743072,2024/12/2,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,test,4d4d2304e9265b029491e1fec60e7431d55285fcad426e...,test,2582720644667933856,2024/10/3,0.0,8.0,8.0,8.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,test,f05c3b1367d78a5da0b060110d8724e72664c86dc456a4...,test,2597987997622206880,2024/10/24,0.0,4.0,4.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,test,d92c8174bf3dd98aa5d11edefef13f24512c53b694071d...,test,2654564558907114656,2025/1/10,0.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,test,f59a8da8330c5d7b1a53654cf76ad1a119e7e9799ed5fc...,test,2596556204524176288,2024/10/22,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96,test,00286615a6d1a01a00e2468184e36494b0622f6eada4be...,test,2597999260377023904,2024/10/24,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
97,test,e5243c068f746779d9df02f90f145ba0c08c14343593de...,test,2614109851667662240,2024/11/15,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98,test,96827ddc764da5e62964ed70359247a1bf10d40dd3a458...,test,2562479522452604064,2024/9/5,0.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
data_tmp.select_dtypes('O')

,name,mobile,idCard,reqToken,backPointTime
0,test,d45735e2015b20c4e8f95645096ae0dac3c9924b9e6b35...,test,2618468494244577696,2024/11/21
1,test,dff94df987f14dd7304d25f291729f0b4926adad0df39b...,test,2626479791808743072,2024/12/2
2,test,4d4d2304e9265b029491e1fec60e7431d55285fcad426e...,test,2582720644667933856,2024/10/3
3,test,f05c3b1367d78a5da0b060110d8724e72664c86dc456a4...,test,2597987997622206880,2024/10/24
4,test,d92c8174bf3dd98aa5d11edefef13f24512c53b694071d...,test,2654564558907114656,2025/1/10
...,...,...,...,...,...
95,test,f59a8da8330c5d7b1a53654cf76ad1a119e7e9799ed5fc...,test,2596556204524176288,2024/10/22
96,test,00286615a6d1a01a00e2468184e36494b0622f6eada4be...,test,2597999260377023904,2024/10/24
97,test,e5243c068f746779d9df02f90f145ba0c08c14343593de...,test,2614109851667662240,2024/11/15
98,test,96827ddc764da5e62964ed70359247a1bf10d40dd3a458...,test,2562479522452604064,2024/9/5


In [7]:
object_columns = []
for column in data_tmp.select_dtypes('O').columns:
    if column not in ['name','idCard','mobile','reqToken','backPointTime','version']:
        print(column)
        object_columns.append(column)

In [8]:
fea_list =list(data_tmp.columns[5:])

In [9]:
fea_list

['TZ_0000_m1',
 'TZ_0000_m12',
 'TZ_0000_m15',
 'TZ_0000_m18',
 'TZ_0000_m2',
 'TZ_0000_m24',
 'TZ_0000_m3',
 'TZ_0000_m4',
 'TZ_0000_m5',
 'TZ_0000_m6',
 'TZ_0000_m9',
 'TZ_0000_w1',
 'TZ_0000_w2',
 'TZ_0000_w3',
 'TZ_0000_w4',
 'TZ_0001_m12',
 'TZ_0001_m15',
 'TZ_0001_m18',
 'TZ_0001_m2',
 'TZ_0001_m24',
 'TZ_0001_m3',
 'TZ_0001_m4',
 'TZ_0001_m5',
 'TZ_0001_m6',
 'TZ_0001_m9',
 'TZ_0001_w2',
 'TZ_0001_w3',
 'TZ_0001_w4',
 'TZ_0002_m12',
 'TZ_0002_m15',
 'TZ_0002_m18',
 'TZ_0002_m2',
 'TZ_0002_m24',
 'TZ_0002_m3',
 'TZ_0002_m4',
 'TZ_0002_m5',
 'TZ_0002_m6',
 'TZ_0002_m9',
 'TZ_0002_w2',
 'TZ_0002_w3',
 'TZ_0002_w4',
 'TZ_0003_m12',
 'TZ_0003_m15',
 'TZ_0003_m18',
 'TZ_0003_m2',
 'TZ_0003_m24',
 'TZ_0003_m3',
 'TZ_0003_m4',
 'TZ_0003_m5',
 'TZ_0003_m6',
 'TZ_0003_m9',
 'TZ_0003_w2',
 'TZ_0003_w3',
 'TZ_0003_w4',
 'TZ_0004_m1',
 'TZ_0004_m12',
 'TZ_0004_m15',
 'TZ_0004_m18',
 'TZ_0004_m2',
 'TZ_0004_m24',
 'TZ_0004_m3',
 'TZ_0004_m4',
 'TZ_0004_m5',
 'TZ_0004_m6',
 'TZ_0004_m9',
 'TZ_

In [10]:
len(fea_list)

15160

In [11]:
if debug:
    fea_list = fea_list[:1000]

In [ ]:
from new_tools import flitter_by_std_fpr
ftr1_keep_dict = {}
length = 500
for i in range(0,(len(fea_list) // length) + 1):
    start,end  = i * length, min((i + 1) * length, len(fea_list))
    print(start, ' - ', end)
    tmp_lst = [i for i in fea_list[start:end] if i not in object_columns]
    with zipfile.ZipFile(DATA_OR_PATH + '驭鉴20250707_LH_deltaV1_result.txt.zip', 'r') as zip_file:
        file_list = zip_file.namelist()
        # print("压缩包内文件:", file_list)
        csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]
        with zip_file.open(csv_filename) as csv_file:
            each_df = pd.read_csv(csv_file, usecols=tmp_lst)
    # each_df = pd.read_csv(DATA_OR_PATH + '驭鉴20250707_LH1__result.txt', usecols=fea_list[start:end])
    ftr1_keep = flitter_by_std_fpr.drop_svr(each_df.fillna(-999), 0.95, tmp_lst)
    ftr1_keep = flitter_by_std_fpr.drop_std(each_df.fillna(-999), 0, ftr1_keep)
    ftr1_keep_dict[i] = ftr1_keep
    del each_df, ftr1_keep
    gc.collect()
    

In [ ]:
fea_list = []
for k,v in ftr1_keep_dict.items():
    fea_list.extend(v)

In [14]:
fea_list = list(set(fea_list))
fea_list

['TZ_0687_m24',
 'TZ_0091_m9',
 'TZ_1401_m3',
 'TZ_0016_w3',
 'TZ_1641_m6',
 'TZ_0124_m5',
 'TZ_0012_m4',
 'TZ_0971_m12',
 'TZ_1191_m1',
 'TZ_1807_m3',
 'TZ_0149_m18',
 'TZ_0171_m5',
 'TZ_1822_m18',
 'TZ_0214_m2',
 'TZ_1661_m6',
 'TZ_0634_m24',
 'TZ_1486_m6',
 'TZ_1212_m6',
 'TZ_1558_m24',
 'TZ_1717_m12',
 'TZ_0895_m24',
 'TZ_0335_m9',
 'TZ_1977_m6',
 'TZ_1850_m6',
 'TZ_0574_w1',
 'TZ_0136_m6',
 'TZ_1263_m12',
 'TZ_1961_m3',
 'TZ_0323_m24',
 'TZ_0577_T3_m3',
 'TZ_0110_m4',
 'TZ_0605_m9',
 'TZ_1475_m24',
 'TZ_1877_m6',
 'TZ_0115_m24',
 'TZ_0434_m8',
 'TZ_1209_m18',
 'TZ_0155_w3',
 'TZ_1936_m1',
 'TZ_1734_m3',
 'TZ_0318_m15',
 'TZ_1615_m12',
 'TZ_1282_m3',
 'TZ_0416_m4',
 'TZ_0320_w3',
 'TZ_0271_m15',
 'TZ_0255_w4',
 'TZ_0279_w2',
 'TZ_0302_m24',
 'TZ_0049_w4',
 'TZ_1867_m3',
 'TZ_0221_w2',
 'TZ_0239_w4',
 'TZ_0454_m12',
 'TZ_0072_m4',
 'TZ_1508_m3',
 'TZ_0327_m15',
 'TZ_0086_w2',
 'TZ_0133_w4',
 'TZ_0650_m6',
 'TZ_0233_m9',
 'TZ_1515_m12',
 'TZ_1535_m24',
 'TZ_1585_m24',
 'TZ_1628_m18',

In [15]:
pd.DataFrame({'var_names': fea_list}).to_csv(DATA_PATH + 'fea_list.csv')

In [ ]:
# fea_list = pd.read_csv(DATA_PATH + 'fea_list.csv')
# fea_list = fea_list['var_names'].tolist()
fea_list

['TZ_0687_m24',
 'TZ_0091_m9',
 'TZ_1401_m3',
 'TZ_0016_w3',
 'TZ_1641_m6',
 'TZ_0124_m5',
 'TZ_0012_m4',
 'TZ_0971_m12',
 'TZ_1191_m1',
 'TZ_1807_m3',
 'TZ_0149_m18',
 'TZ_0171_m5',
 'TZ_1822_m18',
 'TZ_0214_m2',
 'TZ_1661_m6',
 'TZ_0634_m24',
 'TZ_1486_m6',
 'TZ_1212_m6',
 'TZ_1558_m24',
 'TZ_1717_m12',
 'TZ_0895_m24',
 'TZ_0335_m9',
 'TZ_1977_m6',
 'TZ_1850_m6',
 'TZ_0574_w1',
 'TZ_0136_m6',
 'TZ_1263_m12',
 'TZ_1961_m3',
 'TZ_0323_m24',
 'TZ_0577_T3_m3',
 'TZ_0110_m4',
 'TZ_0605_m9',
 'TZ_1475_m24',
 'TZ_1877_m6',
 'TZ_0115_m24',
 'TZ_0434_m8',
 'TZ_1209_m18',
 'TZ_0155_w3',
 'TZ_1936_m1',
 'TZ_1734_m3',
 'TZ_0318_m15',
 'TZ_1615_m12',
 'TZ_1282_m3',
 'TZ_0416_m4',
 'TZ_0320_w3',
 'TZ_0271_m15',
 'TZ_0255_w4',
 'TZ_0279_w2',
 'TZ_0302_m24',
 'TZ_0049_w4',
 'TZ_1867_m3',
 'TZ_0221_w2',
 'TZ_0239_w4',
 'TZ_0454_m12',
 'TZ_0072_m4',
 'TZ_1508_m3',
 'TZ_0327_m15',
 'TZ_0086_w2',
 'TZ_0133_w4',
 'TZ_0650_m6',
 'TZ_0233_m9',
 'TZ_1515_m12',
 'TZ_1535_m24',
 'TZ_1585_m24',
 'TZ_1628_m18',

In [17]:
len(fea_list)

15160

In [18]:
import gc
gc.collect()

0

In [19]:
label = pd.read_csv(DATA_OR_PATH + '22_mashang_200w_20240901_20250131.csv', encoding='gbk')

In [20]:
label.head()

,ms_no,mobile_sha256,hs_date,prod_flag,sample_type,target1_mi,target2_mi,stage
0,2618468494244577696,d45735e2015b20c4e8f95645096ae0dac3c9924b9e6b35...,2024-11-21,S,CA,0.0,0.0,test
1,2626479791808743072,dff94df987f14dd7304d25f291729f0b4926adad0df39b...,2024-12-02,S,CA,0.0,0.0,test
2,2582720644667933856,4d4d2304e9265b029491e1fec60e7431d55285fcad426e...,2024-10-03,S,CA,0.0,0.0,train
3,2597987997622206880,f05c3b1367d78a5da0b060110d8724e72664c86dc456a4...,2024-10-24,S,CA,0.0,0.0,train
4,2654564558907114656,d92c8174bf3dd98aa5d11edefef13f24512c53b694071d...,2025-01-10,S,CA,0.0,NaN,test


In [21]:
from new_tools.reduce_mem_usage import reduce_mem_usage_dic 

def chunk_reduce_mem_usage(file_name,columns):
    chunks = pd.read_csv(file_name, usecols=columns, chunksize=200_000)
    chunk_list = []
    for chunk in chunks:
        chunk_reduce_mem = reduce_mem_usage_dic(chunk)
        del chunk
        chunk_list.append(chunk_reduce_mem)
        del chunk_reduce_mem
    df = pd.concat(chunk_list)
    return df

In [22]:
# from new_tools.reduce_mem_usage import reduce_mem_usage_dic 
# from new_tools import iv_report


# with zipfile.ZipFile(DATA_OR_PATH + '驭鉴20250707_LH_deltaV1_result.txt.zip', 'r') as zip_file:
#     file_list = zip_file.namelist()
#     csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]
    
#     with zip_file.open(csv_filename) as csv_file:
#         result_list2 = []
#         result_list1 = []
#         length = 2000
#         for i in range(0,(len(fea_list) // length) + 1):
#             start, end = i * length, min((i + 1) * length, len(fea_list))
#             print(start, ' - ', end)
#             temp_fea_list = fea_list[start:end]

#             temp_df = chunk_reduce_mem_usage(csv_file, columns=['mobile','backPointTime'] + temp_fea_list)
#             temp_df['backPointTime'] = pd.to_datetime(temp_df['backPointTime']).dt.strftime('%Y-%m-%d')
#             temp_df.rename(columns={'backPointTime': 'hs_date','mobile':'mobile_sha256'}, inplace=True)
#             # temp_df_reduce_mem = reduce_mem_usage_dic(temp_df)
#             # del temp_df
#             temp_data_all = pd.merge(label, temp_df, how='left', on=['mobile_sha256', 'hs_date'])
#             del temp_df
#             temp_data_all['weight'] = 1
#             temp_data_all['target'] = 'train'
#             # temp_data_all.loc[(pd.to_datetime(temp_data_all['backDateTime']) >= pd.to_datetime('2024-06-01')) & (temp_data_all['target'] =='train'),'target'] = 'oot'
#             iv_calculator1 = iv_report.IVCalculator(temp_data_all.fillna(-999),label='target1_mi',target='target',keep_list=temp_fea_list, max_leaf_nodes=6, min_samples_leaf=0.05)
#             iv_calculator2 = iv_report.IVCalculator(temp_data_all.fillna(-999),label='target2_mi',target='target',keep_list=temp_fea_list, max_leaf_nodes=6, min_samples_leaf=0.05)
#             del temp_data_all
#             iv_df1 = iv_calculator1.iv_report(use_thread=True,max_workers=20)
#             iv_df2 = iv_calculator2.iv_report(use_thread=True,max_workers=20)
#             # iv_df['dif'] = iv_df['train_iv']/iv_df['oot_iv']
#             result_list1 = result_list1.append(iv_df1)
#             result_list2 = result_list2.append(iv_df2)
#             del iv_calculator1,iv_df1,iv_calculator2,iv_df2
#             gc.collect()

In [23]:
# from new_tools.reduce_mem_usage import reduce_mem_usage_dic 
# from new_tools import iv_report


# with zipfile.ZipFile(DATA_OR_PATH + '驭鉴20250707_LH_deltaV1_result.txt.zip', 'r') as zip_file:
#     file_list = zip_file.namelist()
#     csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]
    
#     with zip_file.open(csv_filename) as csv_file:
#         result_list2 = []
#         result_list1 = []
#         length = 2000
#         for i in range(0,(len(fea_list) // length) + 1):
#             start, end = i * length, min((i + 1) * length, len(fea_list))
#             print(start, ' - ', end)
#             temp_fea_list = fea_list[start:end]

#             temp_df = pd.read_csv(csv_file, usecols=['mobile','backPointTime'] + temp_fea_list)
#             temp_df['backPointTime'] = pd.to_datetime(temp_df['backPointTime']).dt.strftime('%Y-%m-%d')
#             temp_df.rename(columns={'backPointTime': 'hs_date','mobile':'mobile_sha256'}, inplace=True)
#             temp_df_reduce_mem = reduce_mem_usage_dic(temp_df)
#             del temp_df
#             temp_data_all = pd.merge(label, temp_df_reduce_mem, how='left', on=['mobile_sha256', 'hs_date'])
#             del temp_df_reduce_mem
#             temp_data_all['weight'] = 1
#             temp_data_all['target'] = 'train'
#             # temp_data_all.loc[(pd.to_datetime(temp_data_all['backDateTime']) >= pd.to_datetime('2024-06-01')) & (temp_data_all['target'] =='train'),'target'] = 'oot'
#             iv_calculator1 = iv_report.IVCalculator(temp_data_all.fillna(-999),label='target1_mi',target='target',keep_list=temp_fea_list, max_leaf_nodes=6, min_samples_leaf=0.05)
#             iv_calculator2 = iv_report.IVCalculator(temp_data_all.fillna(-999),label='target2_mi',target='target',keep_list=temp_fea_list, max_leaf_nodes=6, min_samples_leaf=0.05)
#             del temp_data_all
#             iv_df1 = iv_calculator1.iv_report(use_thread=True,max_workers=20)
#             iv_df2 = iv_calculator2.iv_report(use_thread=True,max_workers=20)
#             # iv_df['dif'] = iv_df['train_iv']/iv_df['oot_iv']
#             result_list1 = result_list1.append(iv_df1)
#             result_list2 = result_list2.append(iv_df2)
#             del iv_calculator1,iv_df1,iv_calculator2,iv_df2
#             gc.collect()

In [24]:
object_columns

[]

In [ ]:
# 这个版本是最优版本
# from meu_tools.reduce_menu_usage import reduce_menu_usage_dic
# import pandas as pd
# import zipfile
# import gc

# def chunk_func(name):
#     print("===" * 20)
#     print(name)
    
#     # 读取特征列表
#     fea_list = pd.read_csv(DATA_OR_PATH + f'fea_list_{name}.csv')
#     fea_list = fea_list['var_names'].tolist()
    
#     processed_chunks = []
    
#     # 创建标签索引列表
#     label_index_lst = (label['mobile'] + label['backDateTime'].astype(str)).tolist()
#     label_tmp = label.copy()
    
#     print('first file')
    
#     with zipfile.ZipFile(DATA_PATH + f'联合建模样本_12%_2025805_UI_{name}_result.txt.zip', 'r') as zip_file:
#         file_list = zip_file.namelist()
#         csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]
        
#         with zip_file.open(csv_filename) as csv_file:
#             # 读取分块数据
#             chunks = pd.read_csv(csv_file, usecols=["bucketTime"] + fea_list, chunksize=100000)
            
#             for chunk in chunks:
#                 # 处理数据
#                 temp_df = reduce_menu_usage_dic(chunk)
                
#                 # 删除重复项
#                 temp_df.drop_duplicates(['mobile', 'backDateTime'], inplace=True)
#                 print(1)
                
#                 # 转换时间格式
#                 temp_df['backDateTime'] = pd.to_datetime(temp_df['backDateTime']).dt.strftime('%Y-%m-%d')
#                 temp_df.drop(columns=['backDateTime'], inplace=True)
#                 print(2)
                
#                 # 合并标签数据
#                 temp_df = pd.merge(label_tmp, temp_df, how='inner', on=['mobile', 'backDateTime'])
#                 print(3)
                
#                 # 更新标签索引列表
#                 remove_set = set((temp_df['mobile'] + temp_df['backDateTime'].astype(str)).tolist())
#                 label_index_lst = list(set(label_index_lst) - remove_set)
                
#                 # 过滤标签数据
#                 label_tmp = label[
#                     (label['mobile'] + label['backDateTime'].astype(str)).isin(label_index_lst)
#                 ]
#                 print(4)
                
#                 processed_chunks.append(temp_df)
#                 print(f"Processed chunk: {len(processed_chunks)}")
                
#                 # 清理内存
#                 del chunk, temp_df, remove_set
#                 gc.collect()
#     print('second file')
#     with zipfile.ZipFile(DATA_PATH + f'联合建模样本_id_20250827_LL_{name}_result.txt.zip', 'r') as zip_file:
#         file_list = zip_file.namelist()
#         csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]

#         with zip_file.open(csv_filename) as csv_file:
#             chunks = pd.read_csv(csv_file, usecols=['mobile', 'backPointTime'] + fea_list, chunksize=100000)

#             for chunk in chunks:  # 在 with 块内完成迭代
#                 temp_df = reduce_menu_usage_dic(chunk)
#                 temp_df.drop_duplicates(['mobile', 'backPointTime'], inplace=True)
#                 print(1)

#                 # 转换时间格式并重命名列
#                 temp_df['backPointTime'] = pd.to_datetime(temp_df['backPointTime']).dt.strftime('%Y-%m-%d')
#                 temp_df.rename(columns={'backPointTime': 'backDateTime'}, inplace=True)
#                 print(2)

#                 # 合并数据
#                 temp_df = pd.merge(label_tmp, temp_df, how='inner', on=['mobile', 'backDateTime'])
#                 print(3)

#                 # 更新标签索引列表
#                 remove_set = set((temp_df['mobile'] + temp_df['backDateTime']).tolist())
#                 label_index_lst = list(set(label_index_lst) - remove_set)

#                 # 过滤标签数据
#                 label_tmp = label[
#                     (label['mobile'].astype(str) + label['backDateTime'].astype(str)).isin(label_index_lst)
#                 ]
#                 print(4)

#                 processed_chunks.append(temp_df)
#                 print(5)

#                 # 清理内存
#                 del chunk, temp_df, remove_set
#                 gc.collect()

#     # 处理剩余的mobile数据（没有匹配的样本）
#     mobile_num_df = label[pd.isna(label['mobile'])]  # 或者根据实际逻辑调整
#     mobile_num_df[fea_list] = np.nan
#     processed_chunks.append(mobile_num_df)

#     # 合并所有处理后的数据
#     data_all1 = pd.concat(processed_chunks, ignore_index=True)
#     del processed_chunks
#     gc.collect()

#     print(len(label_index_lst))
#     print(data_all1.shape)
    
#     # you can add some error output if you worry about label_temp remain some keys
#     return data_all1,label_index_lst

In [ ]:
from new_tools.reduce_mem_usage import reduce_mem_usage_dic

processed_chunks = []
label_index_lst = (label['mobile'] + label['backPointTime']).to_list()
label_tmp = label.copy()

with zipfile.ZipFile(DATA_OR_PATH + '信飞20251023_d1_t2_z2_deltaV1_result.zip', 'r') as zip_file:
    file_list = zip_file.namelist()
    csv_filename = [f for f in file_list if f.endswith('.txt') or f.endswith('.csv')][0]
    
    with zip_file.open(csv_filename) as csv_file:
        chunks = pd.read_csv(csv_file, usecols=['mobile','backPointTime'] + fea_list + object_columns, chunksize=200_000)
        i = 1
        for chunk in chunks:  # 在 with 块内完成迭代
            temp_df = reduce_mem_usage_dic(chunk)
            temp_df.drop_duplicates(['mobile', 'backPointTime'], inplace=True)
            print(1)
            temp_df['backPointTime'] = pd.to_datetime(temp_df['backPointTime']).dt.strftime('%Y-%m-%d')            
            print(2)
            temp_df = pd.merge(label_tmp, temp_df, how='inner', on=['mobile', 'backPointTime'])
            print(3)
            remove_set = set((temp_df['mobile'] + temp_df['backPointTime']).to_list())
            label_index_lst = list(set(label_index_lst) - remove_set)
            label_tmp = label[(label['mobile'] + label['backPointTime'].astype(str)).isin(label_index_lst)]
            print(4)
            # processed_chunks.append(temp_df)
            temp_df.to_parquet(DATA_PATH + f'chunk_{i}.parquet')
            print(5)
            del chunk, temp_df, remove_set
            gc.collect()
            i = i+1
        mobile_num_df = label[pd.isna(label['mobile'])]  # 或者根据实际逻辑调整
        mobile_num_df[fea_list] = np.nan
        # processed_chunks.append(mobile_num_df)

print(len(label_index_lst))


当前内存占用: 11567.69 MB


100%|██████████| 15162/15162 [00:07<00:00, 2018.42it/s]


最终内存占用: 5784.61 MB
下降了 50.0%
1
2
3
4
5
当前内存占用: 11567.69 MB


100%|██████████| 15162/15162 [00:07<00:00, 1997.47it/s]


最终内存占用: 5784.61 MB
下降了 50.0%
1
2
3
4
5
当前内存占用: 11567.69 MB


...

最终内存占用: 5784.61 MB
下降了 50.0%
1
2
3
4
5
当前内存占用: 11567.69 MB


100%|██████████| 15162/15162 [00:10<00:00, 1514.12it/s]


最终内存占用: 5784.61 MB
下降了 50.0%
1
2
3
4
5
当前内存占用: 11567.69 MB


100%|██████████| 15162/15162 [00:10<00:00, 1512.45it/s]


最终内存占用: 5784.61 MB
下降了 50.0%
1
2
3
4
5


In [ ]:
data_all1 = pd.concat(processed_chunks, ignore_index=True)
del processed_chunks
gc.collect()
print(data_all1.shape)

In [ ]:
label_index_lst

[]

In [24]:
data_all1.shape

(2000578, 1008)

In [25]:
data_all1.head()

,ms_no,mobile_sha256,hs_date,prod_flag,sample_type,target1_mi,target2_mi,stage,TZ_0000_m1,TZ_0000_m12,...,TZ_0073_m2,TZ_0073_m24,TZ_0073_m3,TZ_0073_m4,TZ_0073_m5,TZ_0073_m6,TZ_0073_m9,TZ_0073_w2,TZ_0073_w3,TZ_0073_w4
0,2618468494244577696,d45735e2015b20c4e8f95645096ae0dac3c9924b9e6b35...,2024-11-21,S,CA,0.0,0.0,test,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
1,2626479791808743072,dff94df987f14dd7304d25f291729f0b4926adad0df39b...,2024-12-02,S,CA,0.0,0.0,test,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2582720644667933856,4d4d2304e9265b029491e1fec60e7431d55285fcad426e...,2024-10-03,S,CA,0.0,0.0,train,0.0,8.0,...,0.0,3.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0
3,2597987997622206880,f05c3b1367d78a5da0b060110d8724e72664c86dc456a4...,2024-10-24,S,CA,0.0,0.0,train,0.0,4.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,0.0,0.0,1.0
4,2654564558907114656,d92c8174bf3dd98aa5d11edefef13f24512c53b694071d...,2025-01-10,S,CA,0.0,NaN,test,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
data_all1['ms_no'] = data_all1['ms_no'].astype('str')

In [27]:
data_all1.to_parquet(DATA_PATH + 'data_all1.parquet')

In [28]:
# data_all1 = pd.read_parquet(DATA_PATH + 'data_all1.parquet')

In [29]:
# data_all1[object_columns] = data_all1[object_columns].apply(
#     lambda x: x.astype('category').cat.codes.replace({-1: np.nan}).astype('float64')
# )

In [30]:
data_all1[object_columns] 

""
0
1
2
3
4
...
2000573
2000574
2000575
2000576


In [31]:
# data_all1[object_columns].count()

In [32]:
data_all1[data_all1['sample_type'] == 'CA'].to_parquet(DATA_PATH + 'data_all1_CA.parquet')

In [33]:
gc.collect()

0

In [34]:
data_all1[data_all1['sample_type'] == 'CR'].to_parquet(DATA_PATH + 'data_all1_CR.parquet')

In [35]:
%reset -f